# Travel Planner Agent — LangGraph Multi-Agent System

A **Coordinator Agent** extracts intent from a free-text travel request and delegates to three
specialized agents:

- **Flights Agent** → Kiwi MCP server (`search_flights`)
- **Venue Agent** → Tavily web search (`search_venues`)
- **Playlist Agent** → SQLite database (`query_playlist_db`)

Before running: `pip install -r requirements.txt` and fill in the real API keys in `.env`
(`GROQ_API_KEY`, `TAVILY_API_KEY`, `LANGCHAIN_API_KEY`). Every tool has a graceful offline
fallback, so the graph still runs end-to-end without live credentials — just with stubbed
tool output.


## 1. Imports

In [15]:
import os
import json
import re
import sqlite3
from typing import TypedDict, List, Optional

from dotenv import load_dotenv

from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import create_react_agent

from tavily import TavilyClient
from langchain_mcp_adapters.client import MultiServerMCPClient


## 2. Load environment variables

In [ ]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

# LangSmith tracing reads the LANGSMITH_* names directly from .env
os.environ.setdefault("LANGSMITH_TRACING", os.getenv("LANGSMITH_TRACING", "false"))
os.environ.setdefault("LANGSMITH_ENDPOINT", os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com"))
os.environ.setdefault("LANGSMITH_PROJECT", os.getenv("LANGSMITH_PROJECT", "travel-agent"))

if not GROQ_API_KEY or GROQ_API_KEY == "your_key_here":
    print("WARNING: GROQ_API_KEY is not set — LLM calls will fail until you add a real key to .env")
if not TAVILY_API_KEY or TAVILY_API_KEY == "your_key_here":
    print("INFO: TAVILY_API_KEY is not set — venue search will use a stub response")
if not LANGSMITH_API_KEY or LANGSMITH_API_KEY == "your_key_here":
    print("INFO: LANGSMITH_API_KEY is not set — LangSmith tracing will be disabled")


## 3. Initialize the LLM (Groq)

In [17]:
# Single shared LLM used by the coordinator (intent extraction) and by every
# specialized agent. Swap the model name for whatever is current in your Groq account.
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY,
)


## 4. Setup tools

### 4a. Flights tool — Kiwi MCP server

In [18]:
KIWI_MCP_URL = "https://mcp.kiwi.com"

mcp_client = MultiServerMCPClient(
    {
        "kiwi": {
            "url": KIWI_MCP_URL,
            "transport": "streamable_http",
        }
    }
)


@tool
def _stub_search_flights(origin: str, destination: str) -> str:
    """Stub flight search used when the Kiwi MCP server is unreachable."""
    return (
        f"[stub] No live Kiwi MCP connection available. "
        f"Would search flights from {origin} to {destination}."
    )


async def load_flights_tool():
    """Fetch the `search_flights` tool from the Kiwi MCP server.

    Falls back to a stub tool if the MCP server is unreachable (no network,
    no auth, etc.) so the rest of the graph can still be exercised offline.
    """
    try:
        tools = await mcp_client.get_tools()
        for t in tools:
            if t.name == "search_flights":
                return t
        raise RuntimeError("search_flights tool not found on Kiwi MCP server")
    except Exception as exc:
        print(f"WARNING: falling back to stub flights tool ({exc})")
        return _stub_search_flights


flights_tool = await load_flights_tool()


### 4b. Venue tool — Tavily web search

In [19]:
tavily_client = (
    TavilyClient(api_key=TAVILY_API_KEY)
    if TAVILY_API_KEY and TAVILY_API_KEY != "your_key_here"
    else None
)


@tool
def search_venues(destination: str, guest_count: int) -> str:
    """Search the web for wedding venues in `destination` that can host `guest_count` guests."""
    query = f"wedding venues in {destination} for {guest_count}"

    if tavily_client is None:
        return f"[stub] No TAVILY_API_KEY configured. Would search: '{query}'"

    try:
        response = tavily_client.search(query=query, max_results=5)
        results = response.get("results", [])
        if not results:
            return f"No venue results found for '{query}'."
        lines = [f"- {r['title']}: {r['url']}" for r in results]
        return f"Top venues for '{query}':\n" + "\n".join(lines)
    except Exception as exc:
        return f"Venue search failed: {exc}"


### 4c. Playlist tool — SQLite database

In [20]:
DB_PATH = "travel_agent.db"


def init_playlist_db():
    """Create the sample `songs` table and seed it if it's empty."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS songs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            artist TEXT NOT NULL,
            genre TEXT NOT NULL
        )
        """
    )
    cur.execute("SELECT COUNT(*) FROM songs")
    if cur.fetchone()[0] == 0:
        sample_songs = [
            ("Take Five", "Dave Brubeck", "jazz"),
            ("So What", "Miles Davis", "jazz"),
            ("Feeling Good", "Nina Simone", "jazz"),
            ("Fly Me to the Moon", "Frank Sinatra", "jazz"),
            ("Blinding Lights", "The Weeknd", "pop"),
            ("Uptown Funk", "Mark Ronson ft. Bruno Mars", "pop"),
            ("Perfect", "Ed Sheeran", "pop"),
            ("Bohemian Rhapsody", "Queen", "rock"),
            ("Sweet Child O' Mine", "Guns N' Roses", "rock"),
            ("Get Lucky", "Daft Punk", "electronic"),
        ]
        cur.executemany(
            "INSERT INTO songs (title, artist, genre) VALUES (?, ?, ?)", sample_songs
        )
        conn.commit()
    conn.close()


init_playlist_db()


@tool
def query_playlist_db(genre: str) -> str:
    """Query the local `songs` SQLite table for tracks matching `genre`."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT title, artist FROM songs WHERE LOWER(genre) = LOWER(?)", (genre,))
    rows = cur.fetchall()
    conn.close()

    if not rows:
        return f"No songs found for genre '{genre}'."

    lines = [f"- {title} - {artist}" for title, artist in rows]
    return f"Playlist for genre '{genre}':\n" + "\n".join(lines)


## 5. Define the specialized agents

Each agent is a small ReAct agent bound to exactly one tool, so it can only do its own job.

In [21]:
FLIGHTS_SYSTEM_PROMPT = (
    "You are a flights specialist agent. Use the search_flights tool to find flight "
    "options between the given origin and destination. Summarize the best options concisely."
)
VENUE_SYSTEM_PROMPT = (
    "You are a venue specialist agent. Use the search_venues tool to find suitable event "
    "venues for the given destination and guest count. Summarize the top options concisely."
)
PLAYLIST_SYSTEM_PROMPT = (
    "You are a playlist specialist agent. Use the query_playlist_db tool to build a short "
    "playlist for the requested music genre."
)

flights_agent = create_react_agent(llm, [flights_tool], prompt=FLIGHTS_SYSTEM_PROMPT)
venue_agent = create_react_agent(llm, [search_venues], prompt=VENUE_SYSTEM_PROMPT)
playlist_agent = create_react_agent(llm, [query_playlist_db], prompt=PLAYLIST_SYSTEM_PROMPT)


/var/folders/hc/s82zhtcs45z6xnn5kmsnrn9m0000gn/T/ipykernel_322/1995839326.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  flights_agent = create_react_agent(llm, [flights_tool], prompt=FLIGHTS_SYSTEM_PROMPT)
/var/folders/hc/s82zhtcs45z6xnn5kmsnrn9m0000gn/T/ipykernel_322/1995839326.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  venue_agent = create_react_agent(llm, [search_venues], prompt=VENUE_SYSTEM_PROMPT)
/var/folders/hc/s82zhtcs45z6xnn5kmsnrn9m0000gn/T/ipykernel_322/1995839326.py:16: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. 

## 6. Define the LangGraph state

The required fields are `origin`, `destination`, `guest_count`, `genre`, `flights_result`,
`venue_result`, `playlist_result`. Three supporting fields are added because the routing and
extraction steps genuinely need them:

- `raw_input` — the original user message, read by `extract_intent_node`
- `intents` — which of flights/venue/playlist the request actually asked for, read by the router
- `extracted_raw` — the coordinator LLM's raw JSON output, handed from `extract_intent_node`
  to `update_state_node`
- `final_output` — the structured `{flights, venue, playlist}` dict produced at the end


In [22]:
class TravelState(TypedDict):
    raw_input: str

    # Required fields
    origin: Optional[str]
    destination: Optional[str]
    guest_count: Optional[int]
    genre: Optional[str]
    flights_result: Optional[str]
    venue_result: Optional[str]
    playlist_result: Optional[str]

    # Supporting fields (routing / plumbing)
    intents: List[str]
    extracted_raw: Optional[str]
    final_output: Optional[dict]


## 7. Define nodes

In [23]:
EXTRACTION_PROMPT = """You are an information-extraction engine for a travel planner.
Given the user's message, extract the following fields and return STRICT JSON only
(no prose, no markdown fences):

{{
  "origin": string or null,
  "destination": string or null,
  "guest_count": integer or null,
  "genre": string or null,
  "intents": array of zero or more of ["flights", "venue", "playlist"]
}}

Infer "intents" from what the user is asking for:
- "flights" if travel/flights between an origin and destination is implied. This includes
  indirect phrasing — e.g. "I'm from X" plus any mention of an event/destination in Y implies
  travel from X to Y, even if the words "flight" or "fly" never appear. If both an origin and a
  destination city can be identified, always include "flights".
- "venue" if an event/venue/wedding location is implied
- "playlist" if music/genre/playlist is implied

User message: {user_input}
"""


async def extract_intent_node(state: TravelState) -> dict:
    """Coordinator step 1: ask the LLM to pull structured fields + intents out of the raw message."""
    prompt = EXTRACTION_PROMPT.format(user_input=state["raw_input"])
    response = await llm.ainvoke([HumanMessage(content=prompt)])
    return {"extracted_raw": response.content}


def _fallback_extraction(raw_input: str) -> dict:
    """Keyword/regex fallback used only if the LLM doesn't return valid JSON."""
    text = raw_input.lower()
    guest_match = re.search(r"(\d+)\s*guests?", text)

    intents = []
    if any(k in text for k in ["flight", "fly", "from "]):
        intents.append("flights")
    if any(k in text for k in ["venue", "wedding", "party", "event"]):
        intents.append("venue")
    if any(k in text for k in ["playlist", "music", "genre", "jazz", "song"]):
        intents.append("playlist")

    return {
        "origin": None,
        "destination": None,
        "guest_count": int(guest_match.group(1)) if guest_match else None,
        "genre": None,
        "intents": intents,
    }


async def update_state_node(state: TravelState) -> dict:
    """Coordinator step 2: parse the extraction JSON and merge fields into shared state."""
    raw = state.get("extracted_raw") or ""
    match = re.search(r"\{.*\}", raw, re.DOTALL)

    try:
        data = json.loads(match.group(0)) if match else json.loads(raw)
    except (json.JSONDecodeError, AttributeError):
        data = _fallback_extraction(state["raw_input"])

    return {
        "origin": data.get("origin"),
        "destination": data.get("destination"),
        "guest_count": data.get("guest_count"),
        "genre": data.get("genre"),
        "intents": data.get("intents") or [],
        "flights_result": None,
        "venue_result": None,
        "playlist_result": None,
    }


async def flights_node(state: TravelState) -> dict:
    """Delegate to the flights_agent with origin + destination."""
    try:
        task = f"Find flights from {state['origin']} to {state['destination']}."
        result = await flights_agent.ainvoke({"messages": [HumanMessage(content=task)]})
        content = result["messages"][-1].content
    except Exception as exc:
        content = f"Flights lookup failed: {exc}"
    return {"flights_result": content}


async def venue_node(state: TravelState) -> dict:
    """Delegate to the venue_agent with destination + guest_count."""
    try:
        task = f"Find wedding venues in {state['destination']} for {state['guest_count']} guests."
        result = await venue_agent.ainvoke({"messages": [HumanMessage(content=task)]})
        content = result["messages"][-1].content
    except Exception as exc:
        content = f"Venue lookup failed: {exc}"
    return {"venue_result": content}


async def playlist_node(state: TravelState) -> dict:
    """Delegate to the playlist_agent with the requested genre."""
    try:
        task = f"Build a short playlist for the genre: {state['genre']}."
        result = await playlist_agent.ainvoke({"messages": [HumanMessage(content=task)]})
        content = result["messages"][-1].content
    except Exception as exc:
        content = f"Playlist lookup failed: {exc}"
    return {"playlist_result": content}


async def final_response_node(state: TravelState) -> dict:
    """Assemble the final structured output."""
    final_output = {
        "flights": state.get("flights_result"),
        "venue": state.get("venue_result"),
        "playlist": state.get("playlist_result"),
    }
    return {"final_output": final_output}


## 8. Define the conditional router

One routing function is reused at every branch point (after `update_state_node`, and after each
specialized node). It walks a fixed pipeline order and returns the next agent whose intent was
requested and hasn't produced a result yet — so the graph naturally skips any agent the user
didn't ask for, and always ends at `final_response_node`.


In [24]:
PIPELINE_ORDER = ["flights", "venue", "playlist"]

NODE_BY_INTENT = {
    "flights": "flights_node",
    "venue": "venue_node",
    "playlist": "playlist_node",
}

RESULT_FIELD_BY_INTENT = {
    "flights": "flights_result",
    "venue": "venue_result",
    "playlist": "playlist_result",
}


def route(state: TravelState) -> str:
    """Return the next node to run, or final_response_node once all requested
    intents have produced a result."""
    for intent in PIPELINE_ORDER:
        already_done = state.get(RESULT_FIELD_BY_INTENT[intent]) is not None
        if intent in state.get("intents", []) and not already_done:
            return NODE_BY_INTENT[intent]
    return "final_response_node"


## 9. Build the graph

In [25]:
graph = StateGraph(TravelState)

graph.add_node("extract_intent_node", extract_intent_node)
graph.add_node("update_state_node", update_state_node)
graph.add_node("flights_node", flights_node)
graph.add_node("venue_node", venue_node)
graph.add_node("playlist_node", playlist_node)
graph.add_node("final_response_node", final_response_node)

graph.add_edge(START, "extract_intent_node")
graph.add_edge("extract_intent_node", "update_state_node")

ROUTE_MAP = {
    "flights_node": "flights_node",
    "venue_node": "venue_node",
    "playlist_node": "playlist_node",
    "final_response_node": "final_response_node",
}

# Same conditional router wired in at every branch point in the pipeline.
graph.add_conditional_edges("update_state_node", route, ROUTE_MAP)
graph.add_conditional_edges("flights_node", route, ROUTE_MAP)
graph.add_conditional_edges("venue_node", route, ROUTE_MAP)
graph.add_conditional_edges("playlist_node", route, ROUTE_MAP)

graph.add_edge("final_response_node", END)


## 10. Compile the graph

In [26]:
app = graph.compile()

# Optional: render the graph structure if graphviz/mermaid deps are installed.
try:
    display(app.get_graph().draw_mermaid_png())
except Exception:
    print(app.get_graph().draw_mermaid())


b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x02\x04\x00\x00\x02\xda\x08\x02\x00\x00\x00\xc1T\xdc\x9e\x00\x00\x10\x00IDATx\x9c\xec\xdd\x05\\T\xd9\xdb\x07\xf034H\t\x82\xd8\xd8\xdd\xdd\x8d\xdd\xb9\xb6k\xc7\xae\xbdv\xeb\xda\x9d\xeb\xbavw\xd7\xda]\xabb\'\x82\x08\x8aJw\xbf?\xb8\xbb\xf3\xf2WD\x18\x98\xfe}\xd7\x0f{\xb9s\xe7N0s\x9es\x9e\xe7\x9e{\x8d\xe2\xe3\xe3\x05\x11\x11\xe97\x03ADDz\x8f\xc1\x80\x88\x88\x18\x0c\x88\x88\x88\xc1\x80\x88\x88\x04\x83\x01\x11\x11\t\x06\x03""\x02#A\xa4*\xb1Q\xe2\x9f\xf3\x01\x1f=\xc2\xc3Cc\xe3b\xe3\xa3"b\xe57\x19\x18\x88\xb8\xb8\x84\x05\x99L\xc4\xc7\xff\xfb\x13\x0c\x8dd\xb11\xf1I7HX\xc0\x8a\xf8\xff_\x93p/\x03\x11/\xdd\xddP\xc4\xff\xb7W\x03#Y\\L\xfcW\x1b|\xb5\x9c\xf8\xab,>.\xde\xc0@\x16\x97\xb0\xdf\xafo5230\x94\xc9L\xcd\x0c\x1cr\x98\x97\xacam\xe3\xc0\xaf\x0c\xe9&\x19\xe7\x19\x90\n\x1c\\\xed\xfd\xd1=\x1c\xcd\xba\x89\x99\x81\xb1\xa9\x81\xa9\x99!\x9a\xf3\xe8\xa8\xffot\xa5\x16Y$\x1f\x0c\xf0\xff\xf8\xffo\xee\xff\x7f\xe1\xdf\xbb\xc8\xef%\xfe\xb7)G+\x8e\x90\xf3\xd5\xfe\xe

## 11. Test execution

In [28]:
initial_state: TravelState = {
    "raw_input": "I'm from Bangalore and I want a hall in CHennai for 100 guests, jazz playlist",
    "origin": None,
    "destination": None,
    "guest_count": None,
    "genre": None,
    "intents": [],
    "extracted_raw": None,
    "flights_result": None,
    "venue_result": None,
    "playlist_result": None,
    "final_output": None,
}

result = await app.ainvoke(initial_state)

print(json.dumps(result["final_output"], indent=2))


{
  "flights": "Based on the search results, there are several flight options available from Bangalore to Chennai. The best options include:\n\n- Indigo: Departing from Bangalore at 8:00 AM and arriving in Chennai at 9:00 AM\n- SpiceJet: Departing from Bangalore at 10:00 AM and arriving in Chennai at 11:00 AM\n- Air India: Departing from Bangalore at 12:00 PM and arriving in Chennai at 1:00 PM\n\nPlease note that the flight schedules and availability may vary depending on the time of year, demand, and other factors. It's always best to check with the airlines for the most up-to-date information and to book your flights in advance to secure the best rates.",
  "venue": "Based on the search results, here are some top wedding venues in Chennai that can accommodate 100 guests:\n1. Rina's Venue - a popular choice for intimate weddings, offering a range of amenities and services.\n2. The Leela Palace Chennai - a luxurious hotel with multiple banquet halls that can host up to 100 guests.\n3. 